<a href="https://colab.research.google.com/github/Pawan-model/Huggingface-Experiment/blob/main/Chapter_05_Datasets/slice_and_dice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install transformers[sentencepiece]

In [ ]:
!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
!unzip drugsCom_raw.zip

In [ ]:
from datasets import load_dataset, Dataset
data_files={'train':'drugsComTrain_raw.tsv','test':'drugsComTest_raw.tsv'}
drug_dataset=load_dataset('csv',data_files=data_files,delimiter='\t')


In [ ]:
drug_sample=drug_dataset['train'].shuffle().select(range(1000))
drug_sample[:3]

In [ ]:
for split in drug_dataset.keys():
  assert len(drug_dataset[split])==len(drug_dataset[split].unique('Unnamed: 0'))

In [ ]:
drug_dataset=drug_dataset.rename_column('Unnamed: 0','patient_id')
drug_dataset=drug_dataset.filter(lambda x: x['condition'] is not None)
def lowercase(example):
  return {'condition': example['condition'].lower()}
drug_dataset.map(lowercase)


In [ ]:
def review_length(example):
  return {'review_length' : len(example['review'].split())}
drug_dataset=drug_dataset.map(review_length)
drug_dataset

In [ ]:
## lets check if any review have short length , we have to remove them , as they does not contribute much
drug_dataset['train'].sort('review_length')[:5]

In [ ]:
# as it contain many review whose lenght is just 1 , so we will remove rows whose columns have lenght less that 30
drug_dataset=drug_dataset.filter(lambda x: x['review_length']>30)
drug_dataset

In [ ]:
## Now , we have to deal the html characters in review
import html
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

In [ ]:
from transformers import AutoTokenizer
checkpoint='bert-base-uncased'
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
#slow_tokenizer=AutoTokenizer.from_pretrained(checkpoint,use_fast=False)
def tokenize_function(example):
  return tokenizer(example['review'],truncation=True)
#def slow_tokenize_function(example):
  #return slow_tokenizer(example['review'],truncation=True)
tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)
#%time tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True)
#%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False)
#%time tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=False)

In [ ]:
drug_dataset
tokenized_dataset

In [ ]:
def tokenize_and_split(example):
  result= tokenizer(example['review'],
                   truncation=True,
                   max_length=128,
                   return_overflowing_tokens=True)
  sample_map = result.pop("overflow_to_sample_mapping")
  for key, values in example.items():
      result[key] = [values[i] for i in sample_map]
  return result
tokenized_dataset=drug_dataset.map(tokenize_and_split,batched=True)


In [ ]:
drug_dataset
tokenized_dataset

Creating Validation set

In [ ]:
drug_dataset_clean=drug_dataset['train'].train_test_split(train_size=0.8,seed=42)
drug_dataset_clean['validation']=drug_dataset_clean.pop('test')
drug_dataset_clean['test']=drug_dataset['test']
drug_dataset_clean